# Train Conv-VAE-Neo

Train BerryPicker's internal single-view convolutional variational autoencoder. The model, data loading, exp/run configuration, and checkpoints are all local to BerryPicker.

In [1]:
import pathlib
import pprint
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import torch
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"
from sensorprocessing.conv_vae_neo import (
    ConvVAENeo, ConvVAENeoLoss, make_dataloaders, seed_everything, train,
)

## Exp/run parameters

In [2]:
creation_style = "exist-ok"
expruns_path = None
results_path = None
epochs = None
experiment = "sensorprocessing_conv_vae_neo"
run = "sp_vae_neo_128_256px"

In [3]:
if expruns_path:
    expruns_path = pathlib.Path(expruns_path)
    if not expruns_path.exists():
        raise FileNotFoundError(expruns_path)
    Config().set_exprun_path(expruns_path)
    Config().copy_experiment(experiment)
    Config().copy_experiment("demonstration")
if results_path:
    results_path = pathlib.Path(results_path)
    if not results_path.exists():
        raise FileNotFoundError(results_path)
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
pprint.pprint(exp)

***ExpRun**: Loading pointer config file:
	/Users/lboloni/.config/BerryPicker/mainsettings.yaml
***ExpRun**: Loading machine-specific config file:
	/Users/lboloni/Google Drive/My Drive/LotziStudy/Code/PackageTracking/BerryPicker/settings/settings-szenes.yaml
***ExpRun**: Using torch device: cpu
***ExpRun**: Configuration for exp/run: sensorprocessing_conv_vae_neo/sp_vae_neo_128_256px successfully loaded
Experiment:
    architecture_version: 1
    base_channels: 32
    batch_size: 16
    bottleneck_max_size: 8
    class: ConvVaeNeoSensorProcessing
    data_dir: /Users/lboloni/Documents/Develop/Data/BerryPicker/sensorprocessing_conv_vae_neo/sp_vae_neo_128_256px
    early_stopping_min_delta: 0.0
    early_stopping_patience: 50
    epochs: 300
    exp_run_sys_indep_file: /Users/lboloni/Documents/Develop/Github/BerryPicker/BerryPicker/src/experiment_configs/sensorprocessing_conv_vae_neo/sp_vae_neo_128_256px.yaml
    experiment_name: sensorprocessing_conv_vae_neo
    frame_stride: 1
    grad

## Train or resume

In [4]:
seed_everything(exp.get("random_seed", 0))
exp.start_timer("training")
try:
    model = train(exp, epochs=epochs)
finally:
    exp.end_timer("training")

***Timer*** training started
***ExpRun**: Configuration for exp/run: demonstration/random-both-cameras-video successfully loaded
***ExpRun**: Configuration for exp/run: demonstration/random-both-cameras-video successfully loaded
***ExpRun**: Configuration for exp/run: demonstration/random-both-cameras-video successfully loaded
***ExpRun**: Configuration for exp/run: demonstration/random-both-cameras-video successfully loaded
***ExpRun**: Configuration for exp/run: demonstration/random-both-cameras-video successfully loaded
***ExpRun**: Configuration for exp/run: demonstration/random-both-cameras-video successfully loaded
Training new model for 300 epochs
  New best model saved with validation loss: 0.0141
Checkpoint saved: /Users/lboloni/Documents/Develop/Data/BerryPicker/sensorprocessing_conv_vae_neo/sp_vae_neo_128_256px/checkpoints/epoch_000001.pth
Epoch [1/300], Train Loss: 0.0326, Val Loss: 0.0141
  New best model saved with validation loss: 0.0116
Checkpoint saved: /Users/lboloni/

KeyboardInterrupt: 

## Inspect validation reconstructions

In [ ]:
device = Config().runtime["device"]
checkpoint_path = pathlib.Path(exp["data_dir"], "checkpoints", "best_model.pth")

# To load an intermediate checkpoint instead, replace checkpoint_path above with:
# checkpoint_path = pathlib.Path(
#     exp["data_dir"], "checkpoints", "epoch_000002.pth"
# )

if not checkpoint_path.is_file():
    raise FileNotFoundError(checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model = ConvVAENeo(exp).to(device)
if "model_state_dict" in checkpoint:
    model_state = checkpoint["model_state_dict"]  # Intermediate checkpoint.
else:
    model_state = checkpoint  # best_model.pth stores the state dict directly.
model.load_state_dict(model_state)
model.eval()
print(f"Loaded checkpoint: {checkpoint_path}")

In [ ]:
_, validation_loader = make_dataloaders(exp)
images = next(iter(validation_loader)).to(Config().runtime["device"])
model.eval()
with torch.no_grad():
    output = model(images)
components = ConvVAENeoLoss(exp).components(output, images)
print({name: float(value) for name, value in components.items()})

reconstructions = output[0].cpu()
originals = images.cpu()
count = min(6, originals.size(0))
fig, axes = plt.subplots(2, count, figsize=(3 * count, 6), squeeze=False)
for index in range(count):
    axes[0, index].imshow(originals[index].permute(1, 2, 0))
    axes[0, index].axis("off")
    axes[1, index].imshow(reconstructions[index].permute(1, 2, 0))
    axes[1, index].axis("off")
axes[0, 0].set_title("Original")
axes[1, 0].set_title("Reconstruction")
fig.tight_layout()
figure_path = pathlib.Path(exp.data_dir(), "training_reconstructions.png")
fig.savefig(figure_path, bbox_inches="tight")
print(f"Saved {figure_path}")
plt.show()